# 03 · 지식그래프와 계층적 장애 시나리오

물리 그래프를 **타입이 있는 이종 그래프**로 재구성한다.
역이 어느 노선·기관·지역에 속하는지를 관계로 표현하면, 단일 그래프로는
표현조차 불가능한 **상위 계층 동시 장애**를 시뮬레이션할 수 있다.

스키마는 `docs/ONTOLOGY.md` 참조.

In [ ]:
# 저장소 루트에서 실행되도록 경로 이동 (notebooks/ 안에서 열었을 때 대비)
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('작업 경로:', os.getcwd())

In [ ]:
import build_kg
build_kg.main()

In [ ]:
import pandas as pd
kgn = pd.read_csv('kg/kg_nodes.csv'); kge = pd.read_csv('kg/kg_edges.csv')
print('엔티티 타입별'); print(kgn['type'].value_counts())
print('\n관계 타입별'); print(kge['predicate'].value_counts())

## 지식그래프 질의 예시

타입 관계를 필터링하는 것만으로 계층적 질문에 답할 수 있다.

In [ ]:
lab = dict(zip(kgn['id'], kgn['label']))

# Q1. 7호선에 속한 역은 몇 개인가
line_id = kgn[(kgn['type']=='Line') & (kgn['label']=='7호선')]['id'].iloc[0]
on = kge[(kge['predicate']=='ON_LINE') & (kge['target']==line_id)]
print('7호선 역수:', len(on))

In [ ]:
# Q2. 운영기관별 '단절 유발 환승역' 수
st = kgn[kgn['type']=='Station'].set_index('id')
op = kge[kge['predicate']=='OPERATED_BY'].set_index('source')['target']
df = st.assign(기관=op.map(lab))
vuln = df[(df['단절유발']==1) & df['환승역'].astype(str).str.contains('환승')]
vuln['기관'].value_counts().head(8)

In [ ]:
# Q3. 시도별 평균 승객가중 영향도
reg = kge[kge['predicate']=='LOCATED_IN'].set_index('source')['target']
df2 = st.assign(시도=reg.map(lab))
df2.groupby('시도')['승객가중효율저하율'].agg(['count','mean']).round(3).sort_values('mean', ascending=False)

## 계층적 장애 시나리오

노선 단위 / 운영기관 단위 / 광역시도 단위 동시 중단을 시뮬레이션한다.

In [ ]:
import kg_scenarios
out = kg_scenarios.run_scenarios()

In [ ]:
out['노선'].head(10)